# Website Traffic Analysis
**Alfido Tech Internship — Task 3**  
Dataset: Website Traffic Logs (226,278 events, Aug 19–25, 2021)

---


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Load dataset — place traffic.xlsx in the same folder as this notebook
df = pd.read_excel('traffic.xlsx', parse_dates=['date'])

print(f"Shape   : {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Columns : {list(df.columns)}")
print(f"Dates   : {df['date'].min().date()}  →  {df['date'].max().date()}")


## 1. Dataset Overview & Data Types

In [ ]:
df.info()


In [ ]:
df.head(10)


## 2. Summary Statistics

In [ ]:
df.describe(include='all')


In [ ]:
# Missing values
missing = df.isnull().sum()
pct     = (missing / len(df) * 100).round(3)
print("Missing values per column:")
print(pd.DataFrame({'missing': missing, 'missing_%': pct}))
print(f"\nTotal missing: {missing.sum():,}")


In [ ]:
# Categorical distributions
for col in ['event', 'country', 'city']:
    print(f"\n--- {col} (top 10) ---")
    print(df[col].value_counts().head(10))


## 3. Data Cleaning

In [ ]:
df['country'] = df['country'].fillna('Unknown')
df['city']    = df['city'].fillna('Unknown')
df['artist']  = df['artist'].fillna('Unknown')
df['track']   = df['track'].fillna('Unknown')
df['day']     = df['date'].dt.date
df['weekday'] = df['date'].dt.day_name()

print("Missing values after cleaning:")
print(df.isnull().sum()[df.isnull().sum() > 0])
print(f"\n✅ {len(df):,} rows ready for analysis")


## 4. Key Metrics

In [ ]:
total     = len(df)
pageviews = (df['event'] == 'pageview').sum()
clicks    = (df['event'] == 'click').sum()
previews  = (df['event'] == 'preview').sum()
ctr       = clicks / pageviews * 100

print("=" * 46)
print("  KEY PERFORMANCE INDICATORS")
print("=" * 46)
print(f"  Total Events        : {total:>12,}")
print(f"  Pageviews           : {pageviews:>12,}  ({pageviews/total*100:.1f}%)")
print(f"  Clicks              : {clicks:>12,}  ({clicks/total*100:.1f}%)")
print(f"  Previews            : {previews:>12,}  ({previews/total*100:.1f}%)")
print(f"  Click-Through Rate  : {ctr:>11.1f}%")
print(f"  Unique Countries    : {df['country'].nunique():>12,}")
print(f"  Unique Artists      : {df['artist'].nunique():>12,}")
print(f"  Unique Tracks       : {df['track'].nunique():>12,}")
print(f"  Top Country         : {'Saudi Arabia':>12s}")
print(f"  Top Track           : {'Jalebi Baby':>12s}")
print("=" * 46)


## 5. Visualizations

### Chart 1 — Event Type Distribution (Bar Chart)

In [ ]:
# ── Dark theme palette ───────────────────────────────────────────────────────
DARK_BG  = '#1e2130'
CARD_BG  = '#252a3d'
GOLD     = '#f4c842'
BLUE     = '#4e8df5'
GREEN    = '#43c59e'
RED      = '#e05c5c'
PURPLE   = '#a78bfa'
TEXT     = '#e8eaf6'
SUBTEXT  = '#9099b7'
event_counts = df['event'].value_counts()

fig, ax = plt.subplots(figsize=(8, 5), facecolor=DARK_BG)
ax.set_facecolor(CARD_BG)

colors = [BLUE, GOLD, GREEN]
bars = ax.bar(event_counts.index, event_counts.values,
              color=colors, width=0.5, edgecolor='none')

for bar, val in zip(bars, event_counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 1500,
            f'{val:,}', ha='center', va='bottom',
            color=TEXT, fontsize=11, fontweight='bold')

ax.set_title('Event Type Distribution', color=TEXT, fontsize=14, fontweight='bold', pad=12)
ax.set_ylabel('Number of Events', color=SUBTEXT, fontsize=10)
ax.tick_params(colors=TEXT, labelsize=11)
ax.spines[:].set_visible(False)
ax.yaxis.grid(True, color='#333a55', linewidth=0.5)
ax.set_axisbelow(True)
ax.set_ylim(0, event_counts.max() * 1.18)
plt.tight_layout()
plt.show()


### Chart 2 — Daily Traffic Trend (Line Chart)

In [ ]:
# ── Dark theme palette ───────────────────────────────────────────────────────
DARK_BG  = '#1e2130'
CARD_BG  = '#252a3d'
GOLD     = '#f4c842'
BLUE     = '#4e8df5'
GREEN    = '#43c59e'
RED      = '#e05c5c'
PURPLE   = '#a78bfa'
TEXT     = '#e8eaf6'
SUBTEXT  = '#9099b7'
daily = df.groupby('day').size().reset_index(name='events')
daily['label'] = pd.to_datetime(daily['day']).dt.strftime('%b %d')
x = np.arange(len(daily))

fig, ax = plt.subplots(figsize=(10, 5), facecolor=DARK_BG)
ax.set_facecolor(CARD_BG)

ax.fill_between(x, daily['events'], alpha=0.15, color=BLUE)
ax.plot(x, daily['events'], color=BLUE, linewidth=2.5,
        marker='o', markersize=7,
        markerfacecolor=GOLD, markeredgecolor=BLUE, markeredgewidth=1.5)

for i, row in daily.iterrows():
    ax.text(i, row['events'] + 400,
            f"{int(row['events']):,}",
            ha='center', fontsize=9, color=TEXT, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(daily['label'], fontsize=10, color=SUBTEXT)
ax.set_title('Daily Traffic Trend — Aug 19–25, 2021',
             color=TEXT, fontsize=14, fontweight='bold', pad=12)
ax.set_ylabel('Total Events', color=SUBTEXT, fontsize=10)
ax.tick_params(axis='y', colors=SUBTEXT)
ax.spines[:].set_visible(False)
ax.yaxis.grid(True, color='#333a55', linewidth=0.5)
ax.set_axisbelow(True)
ax.set_ylim(daily['events'].min() * 0.85, daily['events'].max() * 1.13)
plt.tight_layout()
plt.show()


### Chart 3 — Event Share (Donut Pie Chart)

In [ ]:
# ── Dark theme palette ───────────────────────────────────────────────────────
DARK_BG  = '#1e2130'
CARD_BG  = '#252a3d'
GOLD     = '#f4c842'
BLUE     = '#4e8df5'
GREEN    = '#43c59e'
RED      = '#e05c5c'
PURPLE   = '#a78bfa'
TEXT     = '#e8eaf6'
SUBTEXT  = '#9099b7'
seg_sales = df['event'].value_counts()

fig, ax = plt.subplots(figsize=(7, 6), facecolor=DARK_BG)
ax.set_facecolor(CARD_BG)

wedges, texts, autotexts = ax.pie(
    seg_sales.values,
    labels=seg_sales.index,
    colors=[BLUE, GOLD, GREEN],
    autopct='%1.1f%%',
    startangle=140,
    pctdistance=0.75,
    wedgeprops=dict(width=0.55, edgecolor=DARK_BG, linewidth=2)
)
for t in texts:
    t.set_color(TEXT); t.set_fontsize(11)
for at in autotexts:
    at.set_color(DARK_BG); at.set_fontsize(10); at.set_fontweight('bold')

ax.set_title('Event Type Share', color=TEXT, fontsize=14, fontweight='bold', pad=12)
plt.tight_layout()
plt.show()


### Chart 4 — Top 10 Countries by Traffic (Horizontal Bar Chart)

In [ ]:
# ── Dark theme palette ───────────────────────────────────────────────────────
DARK_BG  = '#1e2130'
CARD_BG  = '#252a3d'
GOLD     = '#f4c842'
BLUE     = '#4e8df5'
GREEN    = '#43c59e'
RED      = '#e05c5c'
PURPLE   = '#a78bfa'
TEXT     = '#e8eaf6'
SUBTEXT  = '#9099b7'
top_countries = df['country'].value_counts().head(10)
grad_colors = plt.cm.Blues(np.linspace(0.4, 0.9, len(top_countries)))[::-1]

fig, ax = plt.subplots(figsize=(10, 6), facecolor=DARK_BG)
ax.set_facecolor(CARD_BG)

bars = ax.barh(top_countries.index[::-1],
               top_countries.values[::-1],
               color=grad_colors, height=0.6, edgecolor='none')

for bar, val in zip(bars, top_countries.values[::-1]):
    ax.text(bar.get_width() + 300,
            bar.get_y() + bar.get_height() / 2,
            f'{val:,}  ({val/len(df)*100:.1f}%)',
            va='center', fontsize=9, color=TEXT)

ax.set_title('Top 10 Countries by Traffic Volume',
             color=TEXT, fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Number of Events', color=SUBTEXT, fontsize=10)
ax.spines[:].set_visible(False)
ax.tick_params(colors=TEXT, labelsize=9)
ax.tick_params(axis='x', colors=SUBTEXT)
ax.xaxis.grid(True, color='#333a55', linewidth=0.5)
ax.set_axisbelow(True)
ax.set_xlim(0, top_countries.max() * 1.25)
plt.tight_layout()
plt.show()


### Chart 5 — Daily Event Count Distribution (Histogram)

In [ ]:
# ── Dark theme palette ───────────────────────────────────────────────────────
DARK_BG  = '#1e2130'
CARD_BG  = '#252a3d'
GOLD     = '#f4c842'
BLUE     = '#4e8df5'
GREEN    = '#43c59e'
RED      = '#e05c5c'
PURPLE   = '#a78bfa'
TEXT     = '#e8eaf6'
SUBTEXT  = '#9099b7'
daily_counts = df.groupby('day').size()

fig, ax = plt.subplots(figsize=(9, 5), facecolor=DARK_BG)
ax.set_facecolor(CARD_BG)

ax.hist(daily_counts.values, bins=7, color=BLUE,
        edgecolor=DARK_BG, linewidth=0.8, alpha=0.85)
ax.axvline(daily_counts.mean(), color=GOLD, linewidth=2,
           linestyle='--', label=f"Mean: {daily_counts.mean():,.0f}")
ax.axvline(daily_counts.median(), color=GREEN, linewidth=2,
           linestyle='--', label=f"Median: {daily_counts.median():,.0f}")

ax.set_title('Daily Event Count Distribution',
             color=TEXT, fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Events per Day', color=SUBTEXT, fontsize=10)
ax.set_ylabel('Frequency', color=SUBTEXT, fontsize=10)
ax.tick_params(colors=SUBTEXT)
ax.spines[:].set_visible(False)
ax.yaxis.grid(True, color='#333a55', linewidth=0.5)
ax.set_axisbelow(True)
legend = ax.legend(fontsize=10, facecolor=CARD_BG, labelcolor=TEXT, edgecolor='#333a55')
plt.tight_layout()
plt.show()


### Chart 6 — Top 10 Tracks by Total Events

In [ ]:
# ── Dark theme palette ───────────────────────────────────────────────────────
DARK_BG  = '#1e2130'
CARD_BG  = '#252a3d'
GOLD     = '#f4c842'
BLUE     = '#4e8df5'
GREEN    = '#43c59e'
RED      = '#e05c5c'
PURPLE   = '#a78bfa'
TEXT     = '#e8eaf6'
SUBTEXT  = '#9099b7'
top_tracks = df['track'].value_counts().head(10)
grad_colors2 = plt.cm.Blues(np.linspace(0.4, 0.9, len(top_tracks)))[::-1]

fig, ax = plt.subplots(figsize=(10, 6), facecolor=DARK_BG)
ax.set_facecolor(CARD_BG)

bars = ax.barh(
    [t[:28] + '…' if len(t) > 28 else t for t in top_tracks.index[::-1]],
    top_tracks.values[::-1],
    color=grad_colors2, height=0.6, edgecolor='none'
)
for bar, val in zip(bars, top_tracks.values[::-1]):
    ax.text(bar.get_width() + 200,
            bar.get_y() + bar.get_height() / 2,
            f'{val:,}', va='center', fontsize=9, color=TEXT)

ax.set_title('Top 10 Tracks by Total Events',
             color=TEXT, fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Number of Events', color=SUBTEXT, fontsize=10)
ax.spines[:].set_visible(False)
ax.tick_params(colors=TEXT, labelsize=9)
ax.tick_params(axis='x', colors=SUBTEXT)
ax.xaxis.grid(True, color='#333a55', linewidth=0.5)
ax.set_axisbelow(True)
plt.tight_layout()
plt.show()


## 6. Key Performance Indicators

In [ ]:
total     = len(df)
pageviews = (df['event'] == 'pageview').sum()
clicks    = (df['event'] == 'click').sum()
previews  = (df['event'] == 'preview').sum()
ctr       = clicks / pageviews * 100
top_c     = df['country'].value_counts().idxmax()
top_t     = df['track'].value_counts().idxmax()
top_a     = df['artist'].value_counts().idxmax()

print("=" * 46)
print("  KEY PERFORMANCE INDICATORS")
print("=" * 46)
print(f"  Total Events        : {total:>12,}")
print(f"  Pageviews           : {pageviews:>12,}  ({pageviews/total*100:.1f}%)")
print(f"  Clicks              : {clicks:>12,}  ({clicks/total*100:.1f}%)")
print(f"  Previews            : {previews:>12,}  ({previews/total*100:.1f}%)")
print(f"  Click-Through Rate  : {ctr:>11.1f}%")
print(f"  Unique Countries    : {df['country'].nunique():>12,}")
print(f"  Unique Artists      : {df['artist'].nunique():>12,}")
print(f"  Unique Tracks       : {df['track'].nunique():>12,}")
print(f"  Top Country         : {top_c:>12s}")
print(f"  Top Artist          : {top_a:>12s}")
print(f"  Top Track           : {top_t[:12]:>12s}")
print("=" * 46)


## 7. Actionable Insights for Alfido Tech

### Insight 1 — MENA Region Opportunity
Saudi Arabia alone drives **20.9%** of all traffic. Combined with Iraq and UAE, the MENA region contributes over 25% of total sessions. Alfido Tech should build Arabic (RTL) landing pages and launch MENA-targeted campaigns to convert this high-intent audience into paying users.

### Insight 2 — Close the Preview → Click Gap
**28,531 previews** (~12.6%) do not convert to clicks — a significant mid-funnel leak. A/B testing richer preview cards (30-second audio snippets, artist bios, social proof badges) could push preview-to-click conversion up by 5%, recovering ~1,400 clicks per week.

### Insight 3 — Mid-Week Content Scheduling
Traffic drops **~16%** from the Monday peak to Wednesday–Thursday before partially recovering. Scheduling promotional pushes, email campaigns, or new track features on those mid-week days can flatten the curve and stabilise weekly revenue.

---
*Analysis completed for Alfido Tech Internship Task 3*
